# Creative OS — Node Engine Prototype

Implements the Hexagonal Blueprint node transition logic from `spec/hexagonal_blueprint_v05.md`.

**Rules enforced:**
- Simultaneous activation limit: max 3 node **types** active at once; exceeding without prior Z fires Lore Creep
- Floor nodes (π, project-elevated H) do not consume a slot
- A tracks per-entity; slot count tracks node type (multiple A instances = 1 A slot)
- **Slot count ≠ state list length.** State line lists all active instances; slot count counts active types.
- Z fires in three modes: Complete, Truncated, Partial/External. Complete may be terminal (entity exits story).
- S deactivation: forcing function and merge-completion are separate events (Ruling 6.3)
- D auto-activates when S deactivates with a shared goal still active (Ruling 6.6 — formalized)

In [ ]:
from typing import Optional


Z_MODES = {'Complete', 'Truncated', 'Partial/External'}


class NodeEngine:
    """Tracks node state against Hexagonal Blueprint rules (v0.5)."""

    def __init__(self, project: str, floor_nodes: Optional[set] = None):
        self.project = project
        self.floor_nodes = floor_nodes or set()
        self.active: dict[str, list[str]] = {}
        self.z_mode: Optional[str] = None
        self.deferred_z: list[dict] = []
        self.violations: list[dict] = []
        self._log: list[dict] = []

    def slot_count(self) -> int:
        return sum(1 for n in self.active if n not in self.floor_nodes)

    def _active_display(self) -> list[str]:
        out = []
        for node, entities in self.active.items():
            tag = '[floor]' if node in self.floor_nodes else ''
            if node == 'A':
                out.extend(f'A({e})' for e in entities)
            else:
                out.append(f'{node}{tag}')
        return out

    def activate(self, node: str, scene: str, entity: str = '_',
                 z_mode: Optional[str] = None) -> None:
        if node != 'Z' and node not in self.floor_nodes:
            if self.slot_count() >= 3:
                v = {'type': 'LORE_CREEP', 'scene': scene, 'node': node,
                     'slot_count': self.slot_count(), 'active': self._active_display()}
                self.violations.append(v)
                print(f'  ⚠  LORE_CREEP at {scene}: {node} would make '
                      f'slot_count={self.slot_count() + 1} without prior Z')

        if node not in self.active:
            self.active[node] = []
        if entity not in self.active[node]:
            self.active[node].append(entity)

        if node == 'Z':
            self.z_mode = z_mode or 'Complete'
            if self.z_mode in ('Truncated', 'Partial/External'):
                self.deferred_z.append({'entity': entity, 'mode': self.z_mode,
                                        'opened_at': scene})
        self._record(scene, 'ACTIVATE', node, entity, z_mode=z_mode)

    def deactivate(self, node: str, scene: str, entity: str = '_',
                   forcing_function: Optional[str] = None,
                   goal_active: bool = False) -> None:
        if node == 'Z':
            self.z_mode = None

        if node in self.active:
            if entity in self.active[node]:
                self.active[node].remove(entity)
            if not self.active[node]:
                del self.active[node]

        self._record(scene, 'DEACTIVATE', node, entity,
                     forcing_function=forcing_function)

        # Ruling 6.6 (formalized): D auto-activates when S deactivates with goal active
        if node == 'S' and goal_active:
            ff = f' [S forcing function: {forcing_function}]' if forcing_function else ''
            print(f'  ▷  D (Unified Pursuit) auto-activated at {scene}{ff}')
            self.activate('D', scene)

    def resolve_deferred_z(self, scene: str, entity: str = '_',
                           in_scene: bool = False) -> None:
        """Resolve a deferred Z for the given entity.

        in_scene=True: only closes Partial/External entries (completed without
        deferral — fires and resolves in the same scene block). Truncated entries
        for the same entity remain open and continue watching.

        in_scene=False (default): closes all deferred entries for the entity
        (full deferred resolution — Ruling 6.4 conditions met).
        """
        if in_scene:
            match = lambda d: d['entity'] == entity and d['mode'] == 'Partial/External'
        else:
            match = lambda d: d['entity'] == entity

        still_open = [d for d in self.deferred_z if not match(d)]
        resolved   = [d for d in self.deferred_z if match(d)]
        self.deferred_z = still_open

        for r in resolved:
            if in_scene:
                print(f'  ✓  Z PARTIAL/EXTERNAL → COMPLETE (in-scene) at {scene}: '
                      f"{r['entity']}")
            else:
                print(f'  ✓  DEFERRED Z RESOLVED at {scene}: '
                      f"{r['entity']} ({r['mode']} from {r['opened_at']}) → Complete")
        self._record(scene, 'Z_RESOLVED', 'Z', entity)

    def report(self, scene: str) -> None:
        z_info = f' | Z:{self.z_mode}' if self.z_mode else ''
        print(f'  {scene:<50} active={self._active_display()}  '
              f'slots={self.slot_count()}{z_info}')

    def _record(self, scene, action, node, entity, z_mode=None, forcing_function=None):
        self._log.append({
            'scene': scene, 'action': action, 'node': node, 'entity': entity,
            'z_mode': z_mode, 'forcing_function': forcing_function,
            'slot_count': self.slot_count(), 'active': self._active_display(),
        })

    def summary(self) -> None:
        print(f'\n── {self.project} Summary ──')
        print(f'  Violations (Lore Creep):      {len(self.violations)}')
        print(f'  Deferred Z still open:         {len(self.deferred_z)}')
        if self.deferred_z:
            for d in self.deferred_z:
                print(f'    → {d["entity"]} ({d["mode"]} opened at {d["opened_at"]})')
        print(f'  D (Unified Pursuit) active:    {"D" in self.active}')
        print(f'  Active nodes at close:         {self._active_display()}')
        print(f'  Slot count at close:           {self.slot_count()}')

## Veritas — Setup

H is elevated to floor status per Ruling 6.1 (the Hum never recedes — premise permanently dissolves structural bounds).

In [ ]:
# Veritas project: H on floor
eng = NodeEngine(project='Veritas', floor_nodes={'H'})

print('Project:', eng.project)
print('Floor nodes:', eng.floor_nodes)
print('π always present (not tracked as slot — structural floor)')

## Trace 001 — Prologue + Chapter 1

In [ ]:
print('── Prologue ──')
eng.activate('W', 'prologue.open')
eng.report('prologue.open')

# Hum arrives — H activates (elevates to floor), W deactivates
eng.activate('H', 'prologue.hum_arrives')  # no slot consumed — floor node
eng.deactivate('W', 'prologue.hum_arrives')  # active tension now detected
eng.report('prologue.hum_arrives')
# γ fires (generative — age of Veritas begins)

print('\n── Chapter 1: Cassie thread ──')
# Dana kitchen scene — A activates (Truth vs. Lie in balance)
eng.activate('A', 'ch1.kitchen_dana', entity='Cassie_domestic')
eng.report('ch1.kitchen_dana')  # slots: A=1

# Car ride — A holds, σ elevated on path
eng.report('ch1.car_ride')

# Boardroom — Z fires (Complete) before slot limit reached
eng.activate('Z', 'ch1.boardroom', z_mode='Complete')  # A + Z = 2 slots
eng.report('ch1.boardroom')
# α, δ, τ fire (duration-dependent — boardroom required full scene)

eng.deactivate('Z', 'ch1.boardroom_close')  # Z deactivates, narrative back in motion
eng.report('ch1.boardroom_close')

print('\n── Chapter 1: Valeria thread ──')
# γ fires (new thread — Valeria introduced)
# A(Valeria) activates — truth vs. necessary lie
eng.activate('A', 'ch1.precinct_intro', entity='Valeria')
eng.report('ch1.precinct_intro')  # A(Cassie_domestic) + A(Valeria) = 1 slot

# Reyes confession — A(precinct) deactivates, A(Valeria) persists
eng.deactivate('A', 'ch1.reyes_confession', entity='Cassie_domestic')
# Note: Cassie_domestic was the stand-in for the precinct's institutional A
# In this trace, precinct A was bundled with Cassie thread — refine in v2
eng.report('ch1.reyes_confession')

# Chapter 1 close — α fires toward Ch.2
eng.report('ch1.close')
print('\nTrace 001 complete.')

## Trace 002 — Chapter 2

In [ ]:
print('── Chapter 2: Cassie thread ──')

# Apartment — A(Cassie+Dana) activates, Z fires (Complete — personal coherence check)
eng.activate('A', 'ch2.apartment_dana', entity='Cassie+Dana')
eng.activate('Z', 'ch2.apartment_z', z_mode='Complete')  # A(Valeria)+A(Cassie+Dana)+Z
eng.report('ch2.apartment_z')  # slots: A=1 (two instances), Z=1 → total 2

# Confession complete — vase shatters, A(Cassie+Dana) resolves, Z deactivates
eng.deactivate('A', 'ch2.vase_shatters', entity='Cassie+Dana')
eng.deactivate('Z', 'ch2.vase_shatters')
eng.report('ch2.vase_shatters')  # slots: A(Valeria)=1

# Office — no Z (no false self remaining to confirm)
# α blocked (spin attempt fails), δ fires when she runs
eng.report('ch2.office_collapse')

print('\n── Chapter 2: Valeria thread ──')

# Precinct — Vance confession
# α, δ fire (cash hits floor, precinct state permanently shifted)
eng.report('ch2.precinct_vance')

# Chapter 2 close — α fires for S (convergence initiated by Valeria reading the text)
eng.activate('S', 'ch2.close_text_received')  # S in transition (α fired)
eng.report('ch2.close_text_received')  # slots: A(Valeria)=1, S=1 → total 2

print('\nTrace 002 complete.')

## Trace 003 — Chapter 3

In [ ]:
print('── Chapter 3: Drive → convergence ──')

# Drive — S in transition, σ at peak, τ fires (duration-dependent approach sequence)
eng.report('ch3.drive_truth_mob')  # slots: A(Valeria)=1, S=1 → 2

# Cassie enters bank — S fully activates (δ fires for S)
eng.report('ch3.cassie_enters_bank')  # slots: A(Valeria)=1, S=1 → 2

# Confrontation — Z fires (Truncated: mob will interrupt before it completes)
# entity='Valeria' so the deferred Z watch list names the right character
eng.activate('Z', 'ch3.confrontation_gun_drawn', entity='Valeria', z_mode='Truncated')
eng.report('ch3.confrontation_gun_drawn')  # A + S + Z = 3 — AT LIMIT

# Mob invades — Z truncated (cut off), S deactivates (merge complete)
# Ruling 6.3: mob is forcing function; merge-completion is the cause
eng.deactivate('Z', 'ch3.mob_invades', entity='Valeria')  # deferred Z watch opened for Valeria
eng.deactivate('S', 'ch3.mob_invades',
               forcing_function='Purified mob',
               goal_active=True)  # goal (Shadow Ledger) still active → null-gap logged
eng.report('ch3.mob_invades')  # slots: A(Valeria)=1

# Smoke grenade — δ fires (Valeria's purpose shifts: arrest → collaborate)
eng.report('ch3.smoke_grenade')

# Tunnel — A(Cassie+Valeria) activates (TILTED — Valeria's armor intact)
# Cassie calls out Valeria on the car. Valeria gets the last word.
eng.activate('A', 'ch3.tunnel_exchange', entity='Cassie+Valeria_tilted')
eng.report('ch3.tunnel_exchange')  # A(Valeria) + A(Cassie+Valeria) = 1 slot

# Chapter close — α fires toward Ch.4, darkness
eng.report('ch3.close_darkness')

print('\nTrace 003 complete.')

## Summary

eng.summary()

print('\n── Deferred Z chain (Veritas) ──')
print('  Ch.1 boardroom:     Complete')
print('  Ch.2 apartment:     Complete')
print('  Ch.3 bank:          Truncated → watching Valeria → resolves Ch.8')
print('  Ch.5 trailer:       Partial/External → Complete (in-scene)')
print('  Ch.8 (pending):     Complete/self-confessed → resolves Ch.3 Truncated')

print('\n── D (Unified Pursuit) ──')
print('  Activated:  ch3.mob_invades (auto — S completed, Ledger+Thorne goal active)')
print('  Still open: pursuing Thorne + stopping Phase 5 through Ch.5 close')
print('  Deactivates: at G (full resolution) or F (resolution approaching)')

print('\n── 3-node limit ──')
peak = max((e['slot_count'] for e in eng._log), default=0)
print(f'  Peak slot count across all traces: {peak}')
print(f'  Lore Creep violations: {len(eng.violations)}')
if not eng.violations:
    print('  ✓ Limit respected across Ch.1–5')

In [ ]:
print('── Chapter 4: Silence of the Lambs ──')
print('Incoming: A(Valeria), A(Cassie+Valeria_tilted), D  |  slots=2')
print('Deferred Z open: Valeria, Truncated (ch3.confrontation_gun_drawn)')
print()

# Tunnel — Cassie confesses PSH role ("Predictive Social Harmonization")
# α fires: Valeria's understanding of what the Hum is begins to shift
# σ elevated: path toward Thorne now named and specific
eng.report('ch4.tunnel_psh_confession')

# City Hall Station — catatonic victims, Operative 4 with HUD
# σ peak: sealed space, actively hunted
# τ fires: chandelier approach sequence (duration-dependent — full arc required)
# 3rd τ instance (Ch.1 boardroom, Ch.3 drive, Ch.4 chandelier — all approach sequences)
eng.report('ch4.station_silent_gallery')
eng.report('ch4.cleaners_emerge')
eng.report('ch4.chandelier_escape')  # τ

# Foreman's trailer — Shadow Ledger biometric auth
# Note: acoustic maps and St. Jude's logs are ALSO staged here in the final manuscript
# but per Defect 5.1 those beats appear as-written in Ch.5 (stitch artifact)
# Tracing as-written: only Thorne.pdf fires in Ch.4
eng.report('ch4.shadow_ledger_open')

# Director_E_Thorne.pdf — Z fires (Complete, entity=Valeria)
# Identity collapse: Thorne mentored her father, shaped her career, ran the trials.
# "she'd been preparing herself to enforce his experiment" (line 746)
# Miller's "Val! We have to go!" arrives at line 747 — AFTER completion. Not Truncated.
# This is institutional-identity collapse — distinct from the Ch.3 moral-act Z (Truncated).
# Ch.3 deferred Z remains open and independent.
eng.activate('Z', 'ch4.thorne_identity_collapse', entity='Valeria', z_mode='Complete')
eng.report('ch4.thorne_identity_collapse')  # A + D + Z = 3 — AT LIMIT

# Z completes in-scene — identity processed, decision made: pursue Thorne
eng.deactivate('Z', 'ch4.z_complete_pursue', entity='Valeria')
eng.report('ch4.z_complete_pursue')  # slots return to 2

# Iron door breached — flee (breaching charge, Cleaners, laser sights)
# Chapter closes at a dead sprint — σ
eng.report('ch4.door_breached_flee')

print()
print('Trace 004 complete.')
print('Z Complete (entity=Valeria) fired and resolved in-scene at Thorne identity collapse.')
print('Peak slot at Z: 3 (A + D + Z). Limit respected.')
print(f'τ: 3rd instance (chandelier approach). All approach sequences so far.')
print('Slot count at close: 2.')
print('Deferred Z: still open — Valeria, Truncated, ch3.confrontation_gun_drawn.')

## Trace 005 — Chapter 5: The Shattered Shield

In [ ]:
print('── Chapter 5: The Shattered Shield ──')
print('Incoming: A(Valeria), A(Cassie+Valeria_tilted), D  |  slots=2')
print('Deferred Z open: Valeria, Truncated (ch3.confrontation_gun_drawn)')
print()
print('MANUSCRIPT DEFECTS LOGGED:')
print('  Defect 5.1 — Scene duplication: Trailer/Toughbook/Shadow-Ledger staged twice.')
print('    Acoustic maps + St. Jude\'s logs appear as-written in Ch.5; belong in Ch.4 post-fix.')
print('    Tracing as-written. Affected scenes annotated below.')
print('  Defect 5.2 — Kinetic break: Ch.4 closes at dead sprint; Ch.5 opens stationary.')
print('    WANDER_ALARM_KINETIC_DEVIATION-class. Framework catch confirmed.')
print()

# Construction site exterior — Miller processing guilt, Cassie + Valeria regroup
# σ active. D persists. Ch.5 proper begins here.
eng.report('ch5.construction_site_exterior')

# [Defect 5.1 — as-written] Foreman's trailer re-staged
# Shadow Ledger opened again. Acoustic maps (Manhattan-Zero.pdf) revealed.
# γ fires: city-as-instrument revelation — new structural information enters system
# Post-fix: this γ belongs in the Ch.4 Toughbook session
eng.report('ch5.acoustic_maps_reveal')  # γ [Defect 5.1]

# [Defect 5.1 — as-written] St. Jude's medical logs
# δ fires: full scope of what Cassie enabled lands
# NOT Z for Cassie — scope expansion, not coherence checkpoint
# Cassie's coherence was confirmed in Ch.3 (bank). This is δ + γ, not Z.
# Post-fix: this δ belongs in the Ch.4 Toughbook session
eng.report('ch5.st_judes_logs')  # δ [Defect 5.1]

# Helicopter arrives — flashbangs, windows shattered
# σ spikes to peak — external threat forces exit
eng.report('ch5.helicopter_flashbangs')

# Crane climb
# τ fires: duration-dependent approach sequence (full climb arc required)
# 4th τ instance (Ch.1, Ch.3, Ch.4, Ch.5 — all approach sequences)
# WATCH: τ at 4/4 approach sequences. Hunt non-approach instance in Ch.6–8 before ruling.
eng.report('ch5.crane_climb')  # τ (4th instance — all approaches)

# Miller — Sonic Projector fires. Hum removed. Clarity arrives.
# Z fires: Complete, TERMINAL
# "he'd wanted to be one of the good guys, and somewhere along the way
#  he'd stopped being good without even noticing" — identity collapse, lines 950–951
# Miller steps deliberately (lines 957–961) — "the choice"
# Valeria's lunge arrives AFTER completion — not Truncated.
# FIRST TERMINAL Z IN CORPUS — entity's death is the resolution.
# Flag opened. Sub-mode ruling held until two more instances.
# D-COMPOSITION EVENT: Miller exits merged unit.
# D persists — goal intact, Valeria+Cassie carry the pursuit forward.
eng.activate('Z', 'ch5.crane_miller_clarity', entity='Miller', z_mode='Complete')
eng.report('ch5.crane_miller_clarity')  # A + D + Z = 3 — AT LIMIT (terminal Complete)

eng.deactivate('Z', 'ch5.crane_miller_falls', entity='Miller')
eng.report('ch5.crane_miller_falls')  # δ (terminal) — slots return to 2

# Truth Bomb detonation — Shadow Ledger broadcast fires
# γ fires: new state created, truth goes global, age of secrets detonated
# No new nodes: story still in motion toward Thorne and Phase 5
# F does not fire: multiple threads still unresolved
eng.report('ch5.truth_bomb')  # γ

# Chapter close — "the truth didn't just hurt. It cleansed."
# δ fires: world-state has shifted
# Z does NOT fire: thematic statement, not character-level identity-collapse
# Z VARIANT WATCH: 2nd instance of thematic chapter-close beat logged as δ
#   (Ch.3 tunnel close + Ch.5 here). One more = ruling threshold met.
# D persists: shared goal (Thorne + stopping Phase 5) still active
eng.report('ch5.close_truth_cleanses')

print()
print('Trace 005 complete.')
print('Defects 5.1 + 5.2 logged. Traced as-written.')
print('Miller Z: Complete, TERMINAL — first terminal Z in corpus. Flag opened.')
print('D-composition event: Miller exits unit. D persists with Valeria+Cassie.')
print('τ: 4/4 approach sequences. Hunt non-approach in Ch.6–8 before ruling.')
print('Z variant watch: 2/3 instances. One more = ruling.')
print('Ch.3 Truncated Z still open — watching Valeria, resolves Ch.8.')

## Summary — Ch.1 through Ch.5

In [ ]:
eng.summary()

print('\n── Deferred Z chain (Veritas) ──')
print('  Ch.1 boardroom:   Complete (Cassie)')
print('  Ch.2 apartment:   Complete (Cassie)')
print('  Ch.3 bank:        Truncated → watching Valeria → resolves Ch.8')
print('  Ch.4 trailer:     Complete (Valeria — Thorne identity collapse)')
print('  Ch.5 crane:       Complete, TERMINAL (Miller — first terminal Z in corpus)')
print('  Ch.8 (pending):   Complete/self-confessed → resolves Ch.3 Truncated')

print('\n── D (Unified Pursuit) ──')
print('  Auto-activated:  ch3.mob_invades (S completed, Ledger+Thorne goal active)')
print('  Composition:     Valeria+Cassie (Miller exited at ch5.crane — D-composition event #1)')
print('  Still running:   through Ch.5 — goal not in sight of resolution')
print('  Deactivates at:  G (full resolution) or F (resolution approaching)')

print('\n── 3-node limit ──')
peak = max((e['slot_count'] for e in eng._log), default=0)
print(f'  Peak slot count Ch.1–5: {peak}')
print(f'  Hit at: Ch.3 bank | Ch.4 trailer (Valeria Z) | Ch.5 crane (Miller Z terminal)')
print(f'  Lore Creep violations:  {len(eng.violations)}')
if not eng.violations:
    print('  ✓ Limit respected throughout')

print('\n── Slot accounting ──')
print('  State line = all active node instances')
print('  Slot count = active node TYPES  (A = 1 slot regardless of instance count)')
print('  H[floor] = 0 slots. 3-node limit applies to slot count, not state list length.')

print('\n── Open watches ──')
print('  τ: 4/4 approach sequences. Hunt non-approach in Ch.6–8 before ruling.')
print('  Z variant (thematic chapter-close): 2/3 instances. One more = ruling.')
print('  Terminal Z: 1 instance (Miller). Two more needed before sub-mode ruling.')
print('  Manuscript: Defect 5.1 (duplication) + 5.2 (kinetic break) at Ch.4→Ch.5 seam.')
print('              Recommendation: collapse dual trailer staging into Ch.4 Toughbook session.')